<a href="https://colab.research.google.com/github/binhminh276/hcmc-house-price_prediction/blob/main/notebooks/01_Crawl_BDS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get update
!wget https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt install -y ./google-chrome-stable_current_amd64.deb
!rm google-chrome-stable_current_amd64.deb

!pip install undetected-chromedriver pandas beautifulsoup4

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,120 kB]
Get:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,613 kB]


In [ ]:
import undetected_chromedriver as uc
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import os
import re

def get_chrome_version():
    try:
        stream = os.popen('/usr/bin/google-chrome --version')
        output = stream.read()
        return int(re.search(r'\d+', output).group(0))
    except Exception as e:
        return None

def setup_driver():
    options = uc.ChromeOptions()
    options.add_argument('--headless')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36')
    #Chặn tải hình ảnh
    options.add_argument('--blink-settings=imagesEnabled=false')
    prefs = {"profile.managed_default_content_settings.images": 2}
    options.add_experimental_option("prefs", prefs)

    chrome_path = '/usr/bin/google-chrome'
    v_main = get_chrome_version()

    driver = uc.Chrome(
        options=options,
        browser_executable_path=chrome_path,
        version_main=v_main,
        headless=True
    )
    return driver

def get_text_safe(element):
    return element.text.strip() if element else "N/A"

1. Cào link

In [ ]:
def scrape_list_pages_large(base_url, start_page, end_page, output_file):
    scraped_pages = set()
    write_header = True
    global_id_counter = 1

    # 1. KIỂM TRA FILE CŨ ĐỂ CÀO TIẾP (RESUME)
    if os.path.exists(output_file):
        try:
            df_existing = pd.read_csv(output_file)
            if 'Trang' in df_existing.columns:
                scraped_pages = set(df_existing['Trang'].dropna().astype(int).tolist())
                print(f"[*] Đã cào {len(scraped_pages)} trang trước đó. Sẽ bỏ qua và cào tiếp...")
                write_header = False
            if 'ID' in df_existing.columns:
                global_id_counter = df_existing['ID'].max() + 1
        except Exception as e:
            print(f"File cũ lỗi hoặc trống: {e}")

    driver = setup_driver()

    try:
        for page in range(start_page, end_page + 1):
            if page in scraped_pages:
                continue

            # Xử lý URL
            current_url = base_url if page == 1 else f"{base_url}/p{page}"
            print(f"\n--- [{page}/{end_page}] Truy cập: {current_url} ---")

            try:
                driver.get(current_url)

                # 2. GIẢ LẬP CON LĂN CHUỘT (SCROLL)
                # Chia làm 4 nhịp cuộn để kích hoạt nạp link chi tiết
                for i in range(1, 5):
                    scroll_to = i * 25
                    driver.execute_script(f"window.scrollTo(0, document.body.scrollHeight * {scroll_to}/100);")
                    time.sleep(random.uniform(1.5, 2.5))

                # Thời gian nghỉ chờ tổng cộng sau khi cuộn
                time.sleep(random.uniform(3, 5))

                soup = BeautifulSoup(driver.page_source, 'html.parser')
                cards = soup.find_all('div', class_='re__card-info')

                if not cards:
                    print(f" -> Cảnh báo: Trang {page} không trả về dữ liệu. Có thể bị block.")
                    # Chụp ảnh lỗi để kiểm tra nếu cần
                    # driver.save_screenshot(f"error_page_{page}.png")
                    continue

                page_data = []
                for card in cards:
                    # Logic lấy link chi tiết từ Code thành công của bạn
                    a_tag = card.find_parent('a', href=True)
                    if not a_tag:
                        h3_title = card.find('h3', class_='re__card-title')
                        if h3_title: a_tag = h3_title.find('a', href=True)
                    if not a_tag:
                        parent_div = card.parent
                        if parent_div: a_tag = parent_div.find('a', href=True)

                    if a_tag and 'href' in a_tag.attrs:
                        raw_href = a_tag['href']
                        detail_url = raw_href if raw_href.startswith('http') else "https://batdongsan.com.vn/nha-dat-ban-tp-hcm" + raw_href
                    else:
                        detail_url = "N/A"

                    # Lấy các thông số cơ bản
                    title = get_text_safe(card.find(class_=lambda x: x and ('pr-title' in x or 're__card-info' in x)))
                    price = get_text_safe(card.find('span', class_='re__card-config-price'))
                    area = get_text_safe(card.find('span', class_='re__card-config-area'))

                    loc_elem = card.find('div', class_='re__card-location')
                    location = loc_elem.find_all('span')[-1].text.strip() if loc_elem and loc_elem.find_all('span') else "N/A"

                    page_data.append({
                            "ID": global_id_counter,
                            "Trang": page,
                            "Tiêu đề": title,
                            "Giá": price,
                            "Diện tích": area,
                            "Vị trí": location,
                            "URL Chi tiết": detail_url
                        })
                    global_id_counter += 1

                # 3. GHI NỐI TIẾP (APPEND) VÀO FILE NGAY LẬP TỨC
                if page_data:
                    df_page = pd.DataFrame(page_data)
                    df_page.to_csv(output_file, mode='a', header=write_header, index=False, encoding='utf-8-sig')
                    write_header = False
                    print(f" -> Thành công: Đã lưu thêm {len(page_data)} tin từ trang {page}.")

                # 4. RESET TRÌNH DUYỆT MỖI 20 TRANG
                if page % 20 == 0:
                    print(" ---> [Hệ thống] Khởi động lại trình duyệt để xả RAM...")
                    driver.quit()
                    time.sleep(5)
                    driver = setup_driver()

            except Exception as e:
                print(f" -> Lỗi tại trang {page}: {e}")
                continue

    finally:
        if driver:
            driver.quit()

    print(f"\n=== HOÀN THÀNH ===\nKết quả lưu tại: {output_file}")
    return pd.read_csv(output_file) if os.path.exists(output_file) else None

URL_GOC = "https://batdongsan.com.vn/nha-dat-ban-tp-hcm"
FILE_LUU = "newDanhSach_BDS.csv"

result = scrape_list_pages_large(URL_GOC, 3000, 3200, FILE_LUU)

[*] Đã cào 50 trang trước đó. Sẽ bỏ qua và cào tiếp...

--- [3000/3200] Truy cập: https://batdongsan.com.vn/nha-dat-ban-tp-hcm/p3000 ---
 -> Thành công: Đã lưu thêm 30 tin từ trang 3000.
 ---> [Hệ thống] Khởi động lại trình duyệt để xả RAM...

--- [3001/3200] Truy cập: https://batdongsan.com.vn/nha-dat-ban-tp-hcm/p3001 ---
 -> Thành công: Đã lưu thêm 30 tin từ trang 3001.

--- [3002/3200] Truy cập: https://batdongsan.com.vn/nha-dat-ban-tp-hcm/p3002 ---
 -> Thành công: Đã lưu thêm 30 tin từ trang 3002.

--- [3003/3200] Truy cập: https://batdongsan.com.vn/nha-dat-ban-tp-hcm/p3003 ---
 -> Thành công: Đã lưu thêm 30 tin từ trang 3003.

--- [3004/3200] Truy cập: https://batdongsan.com.vn/nha-dat-ban-tp-hcm/p3004 ---
 -> Thành công: Đã lưu thêm 30 tin từ trang 3004.

--- [3005/3200] Truy cập: https://batdongsan.com.vn/nha-dat-ban-tp-hcm/p3005 ---
 -> Thành công: Đã lưu thêm 30 tin từ trang 3005.

--- [3006/3200] Truy cập: https://batdongsan.com.vn/nha-dat-ban-tp-hcm/p3006 ---
 -> Thành công:

2. Cào detail

In [ ]:
# --- GIAI ĐOẠN 2: CÀO CHI TIẾT ---
def scrape_detail_urls(csv_input_file, csv_output_file):
    try:
        df_links = pd.read_csv(csv_input_file)
        df_links = df_links.dropna(subset=['URL Chi tiết'])
        df_links = df_links[df_links['URL Chi tiết'].str.startswith('http', na=False)]
    except Exception as e:
        print(f"Lỗi khi đọc file {csv_input_file}: {e}")
        return None

    scraped_ids = set()
    write_header = True

    # CƠ CHẾ RESUME Giai đoạn 2
    if os.path.exists(csv_output_file):
        try:
            df_existing = pd.read_csv(csv_output_file)
            if 'ID' in df_existing.columns:
                scraped_ids = set(df_existing['ID'].dropna().tolist())
                print(f"[*] Đã tìm thấy file lưu chi tiết. Có {len(scraped_ids)} link đã cào trước đó.")
                write_header = False
        except Exception as e:
            print(f"Lỗi đọc file cũ: {e}. Sẽ tạo file mới.")

    start_loc = 2501
    end_loc = 3000
    df_links = df_links[(df_links['ID'] >= start_loc) & (df_links['ID'] <= end_loc)]

    df_links = df_links[~df_links['ID'].isin(scraped_ids)]

    print(f"-> THỰC TẾ SẼ CÀO: {len(df_links)} links thuộc khoảng ID {start_loc}-{end_loc} chưa cào.")

    if len(df_links) == 0:
        print("Đã cào xong toàn bộ danh sách chi tiết.")
        return pd.read_csv(csv_output_file)

    batch_details = []
    driver = None

    try:
        driver = setup_driver()

        for count, (index, row) in enumerate(df_links.iterrows(), start=1):
            bds_id = row.get('ID')
            url = row.get('URL Chi tiết')

            print(f"[{count}/{len(df_links)}] Đang cào ID {bds_id} - URL: {url}")

            detail_data = {
                "ID": bds_id,
                "URL Chi tiết": url,
                "Mô tả": "N/A", "Diện tích": "N/A", "Địa chỉ chi tiết": "N/A",
                "Khoảng giá": "N/A", "Số phòng ngủ": "N/A", "Số phòng tắm, vệ sinh": "N/A",
                "Pháp lý": "N/A", "Nội thất": "N/A", "Ngày hết hạn": "N/A"
            }

            try:
                driver.get(url)
                time.sleep(random.uniform(5, 8)) # Chờ trang detail tải
                soup = BeautifulSoup(driver.page_source, 'html.parser')

                desc_elem = soup.find('div', class_=lambda x: x and 're__detail-content' in x)
                if desc_elem: detail_data['Mô tả'] = desc_elem.text.strip()

                address_elem = soup.find('span', class_='re__address-line-1')
                if address_elem: detail_data['Địa chỉ chi tiết'] = address_elem.text.strip()

                specs_items = soup.find_all('div', class_='re__pr-specs-content-item')
                for item in specs_items:
                    k_elem = item.find('span', class_='re__pr-specs-content-item-title')
                    v_elem = item.find('span', class_='re__pr-specs-content-item-value')
                    if k_elem and v_elem:
                        key = k_elem.text.strip()
                        val = v_elem.text.strip()
                        if "Diện tích" in key: detail_data["Diện tích"] = val
                        elif "Khoảng giá" in key or "Mức giá" in key: detail_data["Khoảng giá"] = val
                        elif "Số phòng ngủ" in key: detail_data["Số phòng ngủ"] = val
                        elif "Số toilet" in key or "phòng tắm" in key.lower() or "vệ sinh" in key.lower(): detail_data["Số phòng tắm, vệ sinh"] = val
                        elif "Pháp lý" in key: detail_data["Pháp lý"] = val
                        elif "Nội thất" in key: detail_data["Nội thất"] = val

                short_infos = soup.find_all('div', class_='re__pr-short-info-item')
                for info in short_infos:
                    k_elem = info.find('span', class_=['title', 're__pr-short-info-item_title'])
                    v_elem = info.find('span', class_=['value', 're__pr-short-info-item_value'])
                    if k_elem and v_elem and "Ngày hết hạn" in k_elem.text.strip():
                        detail_data["Ngày hết hạn"] = v_elem.text.strip()

                batch_details.append(detail_data)

            except Exception as item_error:
                print(f" -> Lỗi bóc tách link ID {bds_id}: {item_error}")
                continue

            # Sao lưu ngay lập tức mỗi 10 links
            if count % 10 == 0 or count == len(df_links):
                df_batch = pd.DataFrame(batch_details)
                df_batch.to_csv(csv_output_file, mode='a', header=write_header, index=False, encoding='utf-8-sig')
                write_header = False
                batch_details = []
                print(f" ---> [Đã lưu] Tiến độ: {count}/{len(df_links)} links...")

            # Khởi động lại trình duyệt mỗi 100 links (Chống crash khi mở 5000 link)
            if count % 100 == 0 and count < len(df_links):
                print(" ---> [Hệ thống] Đang khởi động lại trình duyệt để xả RAM...")
                driver.quit()
                time.sleep(4)
                driver = setup_driver()

    except Exception as e:
        print(f"Lỗi hệ thống: {e}")
    finally:
        if driver:
            driver.quit()
        if batch_details:
            pd.DataFrame(batch_details).to_csv(csv_output_file, mode='a', header=write_header, index=False, encoding='utf-8-sig')

    print(f"\nHOÀN THÀNH GIAI ĐOẠN 2. File chi tiết: {csv_output_file}")
    return pd.read_csv(csv_output_file)

# --- CHẠY GIAI ĐOẠN 2 ---
INPUT_FILE = "newDanhSach_BDS.csv"
OUTPUT_FILE = "newChiTiet_BDS.csv"

if os.path.exists(INPUT_FILE):
    df_chi_tiet = scrape_detail_urls(INPUT_FILE, OUTPUT_FILE)
    if df_chi_tiet is not None:
        display(df_chi_tiet.tail())
else:
    print(f"Chưa tìm thấy {INPUT_FILE}. Vui lòng chạy Cell 1 trước.")

[*] Đã tìm thấy file lưu chi tiết. Có 2500 link đã cào trước đó.
-> THỰC TẾ SẼ CÀO: 500 links thuộc khoảng ID 2501-3000 chưa cào.
[1/500] Đang cào ID 2501 - URL: https://batdongsan.com.vn/nha-dat-ban-tp-hcm/ban-nha-mat-pho-duong-13-phuong-long-binh-71/ban-so-11-uong-13-13-ty-123-5m2-tp-hcm-pr45176664
[2/500] Đang cào ID 2502 - URL: https://batdongsan.com.vn/nha-dat-ban-tp-hcm/ban-can-ho-chung-cu-duong-mai-chi-tho-phuong-thu-thiem-empire-city-thu-thiem/ban-1pn-dung-gu-nguoi-doc-than-cao-cap-thanh-dat-view-song-duoc-san-don-nhat-pr44850086
[3/500] Đang cào ID 2503 - URL: https://batdongsan.com.vn/nha-dat-ban-tp-hcm/ban-shophouse-nha-pho-thuong-mai-duong-phan-van-hon-phuong-tan-thoi-nhat-1-tecco-green-nest/cchu-tuyet-pham-gan-san-bay-tsnhat-cach-10-phut-sieu-re-ep-tai-4-4ty-pr45260073
[4/500] Đang cào ID 2504 - URL: https://batdongsan.com.vn/nha-dat-ban-tp-hcm/ban-nha-rieng-duong-au-co-phuong-9-13-69/ban-hem-144-9-quan-tan-binh-6-11-2-lau-8-7-ty-pr43496972
[5/500] Đang cào ID 2505 - URL: 

,ID,URL Chi tiết,Mô tả,Diện tích,Địa chỉ chi tiết,Khoảng giá,Số phòng ngủ,"Số phòng tắm, vệ sinh",Pháp lý,Nội thất,Ngày hết hạn
2995,2996,https://batdongsan.com.vn/nha-dat-ban-tp-hcm/b...,HOA HỒNG 1%- BÁN NHÀ MTKD ĐƯỜNG 20M - PHƯỜNG T...,72 m²,"Đường Gò Dầu, Phường Tân Sơn Nhì, Quận Tân Phú...","14,5 tỷ",2 phòng,2 phòng,Sổ đỏ/ Sổ hồng,NaN,14/03/2026
2996,2997,https://batdongsan.com.vn/nha-dat-ban-tp-hcm/b...,"Bán nhà hẻm nhựa 12m Bờ Bao Tân Thắng, P. Sơn ...",60 m²,"Đường Bờ Bao Tân Thắng, Phường Sơn Kỳ, Quận Tâ...",7 tỷ,4 phòng,4 phòng,Sổ đỏ/ Sổ hồng,Đầy đủ,14/03/2026
2997,2998,https://batdongsan.com.vn/nha-dat-ban-tp-hcm/b...,Kẹt tiền cần bán căn 2PN - 2WC tại Kingdom 101...,72 m²,"Kingdom 101, Đường Tô Hiến Thành, Phường 14, Q...",7 tỷ,2 phòng,2 phòng,NaN,Đầy đủ,14/03/2026
2998,2999,https://batdongsan.com.vn/nha-dat-ban-tp-hcm/b...,- Nhà mặt tiền kinh doanh gần ngã tư Thủ Đức. ...,200 m²,"Đường Võ Văn Ngân, Phường Linh Chiểu, Thành ph...",29 tỷ,5 phòng,3 phòng,Sổ đỏ/ Sổ hồng,NaN,29/03/2026
2999,3000,https://batdongsan.com.vn/nha-dat-ban-tp-hcm/b...,"NHÀ BÁN QUẬN 4, 5 TẦNG, GẦN ĐƯỜNG, DÂN CƯ CHỈN...",25 m²,"Đường Đoàn Văn Bơ, Phường 10, Quận 4, Hồ Chí Minh",5 tỷ,2 phòng,3 phòng,Sổ đỏ/ Sổ hồng,NaN,14/03/2026
